# 世界通史时空图谱 · 博物馆级历史插图批量工厂（Colab Pro GPU）

利用你的 Google Colab Pro 月度计算点，使用开源 **FLUX.1-schnell**（Apache-2.0，无需 HuggingFace 门禁令牌）或 **SDXL-Turbo** 为图谱全部 **295 项资产**（8 大时代 + 30 个精编国家 + 87 位历史人物 + 170 项重大史事）批量生成统一「古典羊皮纸铜版画 + 博物馆油画」质感的轻量 WebP 插图。

### 推荐算力与耗时参考
- **`L4`（24GB 显存，首选性价比）**：全部 295 张约 **12~15 分钟**，Starter 精选包（74 张）约 **3 分钟**。
- **`A100`（40GB+ 显存）**：全部 295 张约 **6~8 分钟**。
- **`T4`（15GB 显存）**：自动切换至 `SDXL-Turbo` + `fp16-fix VAE`，全部 295 张约 **7~10 分钟**。

### 运行后如何回到本地部署？
最后一步会自动下载 `atlas-illustrations-pack.zip`，回到项目根目录执行一行即可无痛合入：
```bash
npm run media:import -- ~/Downloads/atlas-illustrations-pack.zip
```

In [ ]:
#@title 1. 检查 GPU 显存并设置缓存目录（可选开启 Google Drive 断点续传）
import os, torch

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_SUBDIR = "world-history-atlas/illustrations"  #@param {type:"string"}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ 检测到 GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    gpu_name = "CPU"
    vram_gb = 0.0
    print("⚠️ 未检测到 GPU！请在 Colab 菜单『代码执行程序 -> 更改运行时类型』选择 L4 / A100 / T4 GPU。")

WORK_DIR = "/content/atlas-illustrations"
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = os.path.join("/content/drive/MyDrive", DRIVE_SUBDIR)
        print(f"📁 已绑定 Google Drive 断点缓存目录: {WORK_DIR}")
    except Exception as exc:
        print(f"Drive 挂载跳过，使用临时目录 {WORK_DIR}: {exc}")
else:
    print(f"📁 使用 Colab 本地高速缓存目录: {WORK_DIR}")

os.makedirs(WORK_DIR, exist_ok=True)

In [ ]:
#@title 2. 安装推理依赖 (diffusers + transformers + accelerate + Pillow<13)
!pip -q install --upgrade "diffusers>=0.32.0" "transformers>=4.46.0" "accelerate>=1.0.0" "pillow<13.0,>=10.0" sentencepiece protobuf safetensors matplotlib numpy
print("✅ 依赖安装完成")

In [ ]:
#@title 3. 拉取最新的历史提示词清单 (illustration-prompts.json)
import json, urllib.request

RAW_PROMPTS_URL = "https://raw.githubusercontent.com/quanliulq-pixel/world-history-atlas/main/colab/illustration-prompts.json"  #@param {type:"string"}
LOCAL_PROMPTS_PATH = os.path.join(WORK_DIR, "illustration-prompts.json")

try:
    urllib.request.urlretrieve(RAW_PROMPTS_URL, LOCAL_PROMPTS_PATH)
    print(f"✅ 已从 GitHub 拉取提示词清单: {LOCAL_PROMPTS_PATH}")
except Exception as err:
    print(f"⚠️ 无法直接联网拉取 ({err})，请手动上传 colab/illustration-prompts.json")
    from google.colab import files
    uploaded = files.upload()
    first_name = next(iter(uploaded.keys()))
    with open(LOCAL_PROMPTS_PATH, "wb") as f:
        f.write(uploaded[first_name])

with open(LOCAL_PROMPTS_PATH, "r", encoding="utf-8") as f:
    prompt_catalog = json.load(f)

print("📊 清单统计:", prompt_catalog.get("counts", {}))

In [ ]:
#@title 4. 配置批次与画质，加载开源画图模型
BATCH_MODE = "all"  #@param ["all", "core", "starter"]
MODEL_CHOICE = "auto"  #@param ["auto", "flux-schnell", "sdxl-turbo"]
WEBP_QUALITY = 84  #@param {type:"slider", min:65, max:95, step:1}
SEED_BASE = 42  #@param {type:"integer"}
OVERWRITE_EXISTING = False  #@param {type:"boolean"}
MAX_ITEMS = 0  #@param {type:"integer"}

all_items = prompt_catalog["items"]
if BATCH_MODE == "starter":
    selected_items = [x for x in all_items if x.get("tier") == "starter"]
elif BATCH_MODE == "core":
    selected_items = [x for x in all_items if x.get("tier") in ("starter", "core")]
else:
    selected_items = list(all_items)

if MAX_ITEMS and MAX_ITEMS > 0:
    selected_items = selected_items[:MAX_ITEMS]

if MODEL_CHOICE == "auto":
    use_flux = vram_gb >= 20.0
elif MODEL_CHOICE == "flux-schnell":
    use_flux = True
else:
    use_flux = False

print(f"🎯 本次待生成条目: {len(selected_items)} 张 (模式: {BATCH_MODE})")
print(f"🎨 选用模型: {'FLUX.1-schnell (Apache-2.0)' if use_flux else 'SDXL-Turbo + fp16-fix VAE (T4 稳定高速版)'}")

from diffusers import AutoPipelineForText2Image, AutoencoderKL

if use_flux:
    MODEL_ID = "black-forest-labs/FLUX.1-schnell"
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
    )
    if vram_gb < 36.0:
        pipe.enable_model_cpu_offload()
    else:
        pipe.to("cuda")
else:
    MODEL_ID = "stabilityai/sdxl-turbo"
    try:
        vae = AutoencoderKL.from_pretrained(
            "madebyollin/sdxl-vae-fp16-fix",
            torch_dtype=torch.float16,
        )
        pipe = AutoPipelineForText2Image.from_pretrained(
            MODEL_ID,
            vae=vae,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        )
        print("✅ 已加载 madebyollin/sdxl-vae-fp16-fix（彻底避免 FP16 NaN 黑图/模式异常）")
    except Exception as vae_err:
        print(f"⚠️ fp16-fix VAE 拉取跳过 ({vae_err})，回退为 FP32 VAE。")
        pipe = AutoPipelineForText2Image.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        )
        pipe.vae.to(torch.float32)
    pipe.to("cuda")
    pipe.enable_attention_slicing()

pipe.set_progress_bar_config(disable=True)
print("✅ 模型加载就绪，可以开始批量绘图！")

In [ ]:
#@title 5. 批量生成 WebP 插图并实时写入 manifest.json（支持断点续传）
import datetime, time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

MANIFEST_PATH = os.path.join(WORK_DIR, "manifest.json")
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        manifest = json.load(f)
else:
    manifest = {"version": 1, "model": MODEL_ID, "items": {}}

manifest["model"] = MODEL_ID
manifest_items = manifest.setdefault("items", {})

def save_manifest():
    manifest["generatedAt"] = datetime.datetime.utcnow().isoformat() + "Z"
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

start_ts = time.time()
generated_now = 0
skipped_now = 0
sample_paths = []

for idx, item in enumerate(selected_items, start=1):
    out_rel = item["file"]
    out_abs = os.path.join(WORK_DIR, out_rel)
    os.makedirs(os.path.dirname(out_abs), exist_ok=True)

    if os.path.exists(out_abs) and not OVERWRITE_EXISTING:
        manifest_items[item["key"]] = {
            "file": out_rel,
            "width": item["width"],
            "height": item["height"],
            "aspect": item["aspect"],
        }
        skipped_now += 1
        if len(sample_paths) < 8:
            sample_paths.append((item["label"], out_abs))
        continue

    if use_flux:
        gen_w = 1024 if item["aspect"] == "16:9" else 768
        gen_h = 576 if item["aspect"] == "16:9" else 768
    else:
        gen_w = 768 if item["aspect"] == "16:9" else 512
        gen_h = 448 if item["aspect"] == "16:9" else 512

    generator = torch.Generator(device="cpu").manual_seed(SEED_BASE + idx)
    kwargs = {
        "prompt": item["prompt"],
        "width": gen_w,
        "height": gen_h,
        "num_inference_steps": 4,
        "guidance_scale": 0.0,
        "generator": generator,
    }
    if use_flux:
        kwargs["max_sequence_length"] = 256

    result = pipe(**kwargs)
    raw_img = result.images[0]
    if raw_img.mode != "RGB":
        raw_img = raw_img.convert("RGB")
    img = raw_img.resize((item["width"], item["height"]), Image.Resampling.LANCZOS)

    if generated_now == 0:
        mean_luma = float(np.asarray(img).mean())
        print(f"🔎 首图自检: mode={img.mode}, size={img.size}, mean_brightness={mean_luma:.1f}")
        if mean_luma < 3.0:
            raise RuntimeError(f"首图异常偏黑 (mean_brightness={mean_luma:.2f})，请检查 VAE/dtype 设置！")

    img.save(out_abs, format="WEBP", quality=WEBP_QUALITY, method=6)

    manifest_items[item["key"]] = {
        "file": out_rel,
        "width": item["width"],
        "height": item["height"],
        "aspect": item["aspect"],
    }
    generated_now += 1
    if len(sample_paths) < 8:
        sample_paths.append((item["label"], out_abs))

    if generated_now % 5 == 0:
        save_manifest()
        elapsed = time.time() - start_ts
        avg = elapsed / max(1, generated_now)
        remain_min = ((len(selected_items) - idx) * avg) / 60.0
        print(f"[{idx}/{len(selected_items)}] ✅ {item['label']} | 新绘 {generated_now} 张 | 预计剩余 {remain_min:.1f} 分钟")

save_manifest()
print(f"\n🎉 全部完成！本轮新生成 {generated_now} 张，跳过已有 {skipped_now} 张，manifest 共有 {len(manifest_items)} 项。")

if sample_paths:
    cols = min(4, len(sample_paths))
    rows = (len(sample_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes_list = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for ax in axes_list:
        ax.axis("off")
    for ax, (title, p) in zip(axes_list, sample_paths):
        ax.imshow(Image.open(p))
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 6. 打包导出 atlas-illustrations-pack.zip 并下载到本地
import zipfile

ZIP_PATH = "/content/atlas-illustrations-pack.zip"
file_count = 0

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(MANIFEST_PATH, "manifest.json")
    for key, entry in manifest_items.items():
        rel = entry["file"]
        abs_path = os.path.join(WORK_DIR, rel)
        if os.path.exists(abs_path):
            zf.write(abs_path, rel)
            file_count += 1

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"📦 已打包 {file_count} 张 WebP 插图 -> {ZIP_PATH} ({size_mb:.2f} MB)")

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as exc:
    print(f"请手动从左侧文件栏或 Google Drive 下载 {ZIP_PATH}: {exc}")